# Conexão + criação do schema pgvector

In [ ]:
import psycopg
from psycopg.rows import dict_row

def pg_connect():
    conn = psycopg.connect(
        host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS,
        autocommit=True
    )
    return conn

DDL = f"""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS pip_chunks (
    id UUID PRIMARY KEY,
    disciplina TEXT NOT NULL,
    source TEXT NOT NULL,
    chunk_index INT NOT NULL,
    content TEXT NOT NULL,
    embedding VECTOR({EMB_DIM}) NOT NULL
);

-- Índice IVFFlat para COSINE (recomendado criar após inserir dados)
-- CREATE INDEX IF NOT EXISTS pip_chunks_emb_ivfflat
-- ON pip_chunks USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)

print("OK: extensão/tabela prontas.")

# Leitura de arquivos (igual ao anterior)

In [ ]:
# !pip -q install docling

In [ ]:
import re
from pathlib import Path

from docling.document_converter import DocumentConverter  # Docling
# Quickstart: converter.convert(source).document; doc.export_to_markdown()
# :contentReference[oaicite:2]{index=2}

def clean_text(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

# Converter único (reuso melhora performance)
_converter = DocumentConverter()

def read_with_docling(path: Path) -> str:
    """
    Lê PDF/DOCX via Docling e retorna Markdown (melhor para chunking/RAG).
    Docling suporta vários formatos; aqui restringimos aos necessários.
    """
    doc = _converter.convert(str(path)).document
    md = doc.export_to_markdown()
    return clean_text(md)

def read_txt(path: Path) -> str:
    # TXT é melhor ler direto (Docling suporta, mas não precisa)
    return clean_text(path.read_text(encoding="utf-8", errors="ignore"))

ALLOWED_EXTS = {".pdf", ".docx", ".txt"}

def load_documents(folder: str):
    folder = Path(folder)
    assert folder.exists(), f"Pasta não encontrada: {folder}"

    docs = []
    for p in folder.rglob("*"):
        if p.is_dir():
            continue

        ext = p.suffix.lower()
        if ext not in ALLOWED_EXTS:
            continue

        try:
            if ext == ".txt":
                text = read_txt(p)
            else:
                # PDF/DOCX -> Docling
                text = read_with_docling(p)

            if text.strip():
                docs.append({"path": str(p), "text": text, "ext": ext})

        except Exception as e:
            print("Falha ao ler", p, "->", e)

    return docs

DISCIPLINA_DIR = r"D:\Projetos\professor-in-pocket\data\eng_comp\curso\estrutura_de_dados"
docs = load_documents(DISCIPLINA_DIR)
len(docs), [d["path"] for d in docs[:3]]

# Chunking

In [ ]:
def chunk_text(text: str, chunk_size=900, overlap=150):
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

all_chunks = []
for d in docs:
    chs = chunk_text(d["text"], chunk_size=900, overlap=150)
    for i, ch in enumerate(chs):
        all_chunks.append({
            "id": str(uuid.uuid4()),
            "disciplina": "programacao",
            "source": d["path"],
            "chunk_index": i,
            "content": ch,
        })

len(all_chunks), all_chunks[0]["source"]

# Embeddings (BGE-M3) e INSERT no Postgres

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")  # https://huggingface.co/BAAI/bge-m3

def embed_texts(texts, batch_size=16):
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        v = embedder.encode(batch, normalize_embeddings=True)  # bom para cosine
        vecs.extend(v.tolist())
    return vecs

# Embeda tudo
texts = [c["content"] for c in all_chunks]
vecs = embed_texts(texts, batch_size=16)

# Insere (UPSERT)
UPSERT = """
INSERT INTO pip_chunks (id, disciplina, source, chunk_index, content, embedding)
VALUES (%s::uuid, %s, %s, %s, %s, %s::vector)
ON CONFLICT (id) DO UPDATE SET
  disciplina = EXCLUDED.disciplina,
  source = EXCLUDED.source,
  chunk_index = EXCLUDED.chunk_index,
  content = EXCLUDED.content,
  embedding = EXCLUDED.embedding;
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        for c, v in tqdm(list(zip(all_chunks, vecs)), total=len(all_chunks)):
            cur.execute(UPSERT, (c["id"], c["disciplina"], c["source"], c["chunk_index"], c["content"], v))

print("OK: chunks indexados no Postgres.")

# Criar índice IVFFlat (cosine) + probes

In [ ]:
LISTS = 100  # comece assim; depois você ajusta pelo volume de dados
CREATE_INDEX = f"""
CREATE INDEX IF NOT EXISTS pip_chunks_emb_ivfflat
ON pip_chunks USING ivfflat (embedding vector_cosine_ops) WITH (lists = {LISTS});
"""

with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(CREATE_INDEX)

print("OK: índice IVFFlat (cosine) criado.")